# Portail d'emploi : base de données SQLite et requêtes SQL

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anasschahide05-tech/portail-emploi-sql/blob/main/portail_emploi_sql.ipynb)

Ce notebook contient **tout le travail au même endroit** :

1. le schéma de la base (3 tables) ;
2. la création des tables et les données (5 entreprises, 12 offres, 15 candidats) ;
3. **5 questions SQL**, chacune avec sa requête, son résultat et la réponse commentée.

Base de données en mémoire (`:memory:`) : il suffit d'exécuter les cellules dans l'ordre.

## 1. Schéma de la base

![Schéma relationnel](https://raw.githubusercontent.com/anasschahide05-tech/portail-emploi-sql/main/schema_portail_emploi.png)

```
ENTREPRISE (entreprise_id PK, nom_entreprise UQ, secteur, localisation, nombre_employes, date_creation, site_web UQ)
     │ 1
     │ publie
     │ N
OFFRE (offre_id PK, entreprise_id FK, titre_poste, description, salaire_min, salaire_max,
       niveau_experience, competences_requises, date_publication, date_limite, type_contrat, statut)

PROFIL_CANDIDAT (candidat_id PK, nom_candidat, email UQ, telephone, localisation, date_naissance,
                 niveau_etudes, annees_experience, specialite, competences, cv_telecharge,
                 statut_candidat, date_inscription)
```

Une seule clé étrangère existe : `offre.entreprise_id → entreprise.entreprise_id`. La table `profil_candidat` n'est reliée à aucune autre : on ne peut la rapprocher des offres que par la ville (`localisation`).

## 2. Connexion et création des tables

In [1]:
import sqlite3
import pandas as pd
from datetime import datetime, date

# Créer ou se connecter à la base de données
conn = sqlite3.connect(':memory:')  # Base de données en mémoire
cursor = conn.cursor()

print("✓ Connexion à SQLite établie")
print(f"✓ Base de données créée en mémoire")
print(f"✓ Date/Heure: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✓ Connexion à SQLite établie
✓ Base de données créée en mémoire
✓ Date/Heure: 2026-10-22 10:32:25


In [2]:
# Table ENTREPRISE
cursor.execute('''
    CREATE TABLE entreprise (
        entreprise_id INTEGER PRIMARY KEY AUTOINCREMENT,
        nom_entreprise VARCHAR(100) NOT NULL UNIQUE,
        secteur VARCHAR(50) NOT NULL,
        localisation VARCHAR(100) NOT NULL,
        nombre_employes INTEGER CHECK (nombre_employes >= 1),
        date_creation DATE DEFAULT CURRENT_DATE,
        site_web VARCHAR(100) UNIQUE
    )
''')

# Table OFFRE
cursor.execute('''
    CREATE TABLE offre (
        offre_id INTEGER PRIMARY KEY AUTOINCREMENT,
        entreprise_id INTEGER NOT NULL,
        titre_poste VARCHAR(100) NOT NULL,
        description TEXT,
        salaire_min DECIMAL(10,2) CHECK (salaire_min >= 0),
        salaire_max DECIMAL(10,2) CHECK (salaire_max >= 0),
        niveau_experience VARCHAR(50) NOT NULL,
        competences_requises VARCHAR(200),
        date_publication DATE DEFAULT CURRENT_DATE,
        date_limite DATE,
        type_contrat VARCHAR(50) NOT NULL,
        statut VARCHAR(50) DEFAULT 'Ouvert',
        FOREIGN KEY (entreprise_id) REFERENCES entreprise(entreprise_id)
    )
''')

# Table PROFIL_CANDIDAT
cursor.execute('''
    CREATE TABLE profil_candidat (
        candidat_id INTEGER PRIMARY KEY AUTOINCREMENT,
        nom_candidat VARCHAR(100) NOT NULL,
        email VARCHAR(100) UNIQUE NOT NULL,
        telephone VARCHAR(20),
        localisation VARCHAR(100) NOT NULL,
        date_naissance DATE,
        niveau_etudes VARCHAR(50) NOT NULL,
        annees_experience INTEGER CHECK (annees_experience >= 0),
        specialite VARCHAR(100),
        competences VARCHAR(200),
        cv_telecharge DATE,
        statut_candidat VARCHAR(50) DEFAULT 'Actif',
        date_inscription DATE DEFAULT CURRENT_DATE
    )
''')

conn.commit()
print("✓ Table ENTREPRISE créée")
print("✓ Table OFFRE créée")
print("✓ Table PROFIL_CANDIDAT créée")

✓ Table ENTREPRISE créée
✓ Table OFFRE créée
✓ Table PROFIL_CANDIDAT créée


## 3. Insertion des données (5 entreprises, 12 offres, 15 candidats)

In [3]:
# Insérer les entreprises
entreprises = [
    ('TechVision Maroc', 'Tech', 'Casablanca', 150, '2015-03-10', 'www.techvision.ma'),
    ('FinanceHub', 'Finance', 'Rabat', 85, '2018-05-20', 'www.financehub.ma'),
    ('MediCare Plus', 'Santé', 'Fes', 200, '2010-01-15', 'www.medicareplus.ma'),
    ('RetailMax', 'Commerce', 'Marrakech', 320, '2005-06-30', 'www.retailmax.ma'),
    ('EduTech Solutions', 'Éducation', 'Casablanca', 50, '2020-09-08', 'www.edutech.ma')
]

cursor.executemany('''
    INSERT INTO entreprise (nom_entreprise, secteur, localisation, nombre_employes, date_creation, site_web)
    VALUES (?, ?, ?, ?, ?, ?)
''', entreprises)

print(f"✓ {len(entreprises)} entreprises insérées")

✓ 5 entreprises insérées


In [4]:
# Insérer les offres
offres = [
    (1, 'Développeur Python', 'Cherchons dev Python expérimenté', 25000, 35000, 'Confirmé', 'Python;Django;SQL', '2025-10-01', '2025-10-31', 'CDI', 'Ouvert'),
    (1, 'Junior Developer', 'Jeune diplômé bienvenu', 15000, 20000, 'Débutant', 'Python;HTML;CSS', '2025-10-02', '2025-10-15', 'CDI', 'Ouvert'),
    (2, 'Analyste Financier', 'Analyse de données financières', 30000, 40000, 'Expert', 'Excel;SQL;Power BI', '2025-09-15', '2025-10-20', 'CDI', 'Ouvert'),
    (2, 'Assistant Comptable', 'Support comptable', 12000, 16000, 'Débutant', 'Excel;Comptabilité', '2025-10-05', '2025-10-25', 'CDD', 'Ouvert'),
    (3, 'Infirmier Spécialisé', 'Unité de soins', 18000, 24000, 'Confirmé', 'Nursing;Diagnostique', '2025-09-20', '2025-10-15', 'CDI', 'Fermé'),
    (3, 'Aide-Soignant', 'Support médical', 10000, 13000, 'Débutant', 'Premiers secours', '2025-10-03', '2025-10-31', 'CDI', 'Ouvert'),
    (4, 'Chef de Rayon', 'Gestion rayon retail', 16000, 22000, 'Confirmé', 'Management;Retail', '2025-10-01', '2025-10-20', 'CDI', 'Ouvert'),
    (4, 'Vendeur', 'Accueil clients', 8000, 12000, 'Débutant', 'Vente;Communication', '2025-10-02', '2025-11-05', 'CDI', 'Pourvu'),
    (5, 'Responsable Pédagogique', 'Gestion cursus', 20000, 28000, 'Confirmé', 'Pédagogie;Gestion', '2025-10-01', '2025-10-31', 'CDI', 'Ouvert'),
    (5, 'Formateur IT', 'Cours informatique', 18000, 25000, 'Confirmé', 'IT;Pédagogie', '2025-10-04', '2025-10-31', 'CDD', 'Ouvert'),
    (1, 'Data Analyst', 'Analyse de données', 28000, 38000, 'Expert', 'Python;SQL;Tableau', '2025-09-25', '2025-10-20', 'CDI', 'Ouvert'),
    (2, 'Stagiaire Finance', 'Stage 6 mois', 6000, 8000, 'Débutant', 'Excel;Bases finance', '2025-10-10', '2025-11-10', 'Stage', 'Ouvert')
]

cursor.executemany('''
    INSERT INTO offre (entreprise_id, titre_poste, description, salaire_min, salaire_max,
                       niveau_experience, competences_requises, date_publication, date_limite,
                       type_contrat, statut)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
''', offres)

print(f"✓ {len(offres)} offres insérées")

✓ 12 offres insérées


In [5]:
# Insérer les candidats
candidats = [
    ('Ahmed Bennani', 'ahmed.b@email.com', '0612345678', 'Casablanca', '1992-05-15', 'Master', 5, 'Développement web', 'Python;Django;JavaScript', '2025-09-20', 'Actif', '2025-01-10'),
    ('Fatima Zahra Alaoui', 'fatima.z@email.com', '0698765432', 'Fes', '1995-08-22', 'Licence', 3, 'Data Science', 'Python;SQL;Tableau', '2025-09-18', 'Actif', '2025-02-15'),
    ('Mohammed Karim', 'm.karim@email.com', '0645670890', 'Rabat', '1998-11-30', 'Licence', 1, 'Infirmerie', 'Nursing;Premiers secours', '2025-09-25', 'Actif', '2025-03-01'),
    ('Laila Saada', 'l.saada@email.com', '0632145678', 'Marrakech', '1990-03-10', 'Master', 8, 'Finance', 'Excel;Power BI;SQL', '2025-09-22', 'Actif', '2025-04-12'),
    ('Youssef Tahir', 'y.tahir@email.com', '0656789012', 'Agadir', '1999-07-05', 'Bac', 0, 'Vente-Commerce', 'Vente;Communication', '2025-09-28', 'Actif', '2025-05-20'),
    ('Nadia Hassan', 'nadia.h@email.com', '0634567890', 'Casablanca', '1993-02-18', 'Master', 6, 'Gestion RH', 'Management;Pédagogie', '2025-09-20', 'Actif', '2025-06-10'),
    ('Ibrahim Moussaoui', 'ibrahim.m@email.com', '0667890123', 'Rabat', '1996-09-12', 'Licence', 2, 'Informatique', 'Python;Java;HTML', '2025-09-19', 'Actif', '2025-07-05'),
    ('Salima Bouazza', 'salima.b@email.com', '0623456789', 'Fes', '1994-12-25', 'Licence', 4, 'Infirmerie', 'Nursing;Diagnostique', '2025-09-24', 'Actif', '2025-08-15'),
    ('Khalid Amrani', 'khalid.a@email.com', '0645678901', 'Marrakech', '1997-06-08', 'Licence', 2, 'Comptabilité', 'Excel;Comptabilité', '2025-09-21', 'Inactif', '2025-09-01'),
    ('Zainab Bennani', 'zainab.b@email.com', '0612398765', 'Casablanca', '1992-04-30', 'Master', 7, 'Développement', 'Python;SQL;JavaScript', '2025-09-26', 'Actif', '2025-09-10'),
    ('Marwan Leblanc', 'marwan.l@email.com', '0656712345', 'Rabat', '1995-01-14', 'Licence', 3, 'IT', 'Java;Python;SQL', '2025-09-20', 'Actif', '2025-09-15'),
    ('Aisha Farraj', 'aisha.f@email.com', '0634598765', 'Casablanca', '1991-10-22', 'Master', 9, 'Pédagogie', 'Pédagogie;IT;Gestion', '2025-09-25', 'Actif', '2025-09-20'),
    ('Tariq El Fassi', 'tariq.ef@email.com', '0667543210', 'Fes', '1998-08-17', 'Licence', 1, 'Santé', 'Premiers secours;Nursing', '2025-09-23', 'Actif', '2025-09-22'),
    ('Sara Belkadi', 'sara.b@email.com', '0623784590', 'Marrakech', '1993-05-11', 'Licence', 4, 'Retail', 'Management;Vente', '2025-09-27', 'Bloqué', '2025-09-25'),
    ('Rashid Mediane', 'rashid.m@email.com', '0645932187', 'Agadir', '1994-11-09', 'Licence', 5, 'Développement', 'Python;Django', '2025-09-22', 'Actif', '2025-09-28')
]

cursor.executemany('''
    INSERT INTO profil_candidat (nom_candidat, email, telephone, localisation, date_naissance,
                                niveau_etudes, annees_experience, specialite, competences,
                                cv_telecharge, statut_candidat, date_inscription)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
''', candidats)

conn.commit()
print(f"✓ {len(candidats)} candidats insérés")
print(f"\n✓ Total: {len(entreprises)} + {len(offres)} + {len(candidats)} = {len(entreprises) + len(offres) + len(candidats)} enregistrements")

✓ 15 candidats insérés

✓ Total: 5 + 12 + 15 = 32 enregistrements


### Vérification des données

In [6]:
pd.read_sql_query("SELECT * FROM entreprise", conn)

,entreprise_id,nom_entreprise,secteur,localisation,nombre_employes,date_creation,site_web
0,1,TechVision Maroc,Tech,Casablanca,150,2015-03-10,www.techvision.ma
1,2,FinanceHub,Finance,Rabat,85,2018-05-20,www.financehub.ma
2,3,MediCare Plus,Santé,Fes,200,2010-01-15,www.medicareplus.ma
3,4,RetailMax,Commerce,Marrakech,320,2005-06-30,www.retailmax.ma
4,5,EduTech Solutions,Éducation,Casablanca,50,2020-09-08,www.edutech.ma


In [7]:
pd.read_sql_query("SELECT offre_id, entreprise_id, titre_poste, salaire_min, salaire_max, niveau_experience, type_contrat, statut FROM offre", conn)

,offre_id,entreprise_id,titre_poste,salaire_min,salaire_max,niveau_experience,type_contrat,statut
0,1,1,Développeur Python,25000,35000,Confirmé,CDI,Ouvert
1,2,1,Junior Developer,15000,20000,Débutant,CDI,Ouvert
2,3,2,Analyste Financier,30000,40000,Expert,CDI,Ouvert
3,4,2,Assistant Comptable,12000,16000,Débutant,CDD,Ouvert
4,5,3,Infirmier Spécialisé,18000,24000,Confirmé,CDI,Fermé
5,6,3,Aide-Soignant,10000,13000,Débutant,CDI,Ouvert
6,7,4,Chef de Rayon,16000,22000,Confirmé,CDI,Ouvert
7,8,4,Vendeur,8000,12000,Débutant,CDI,Pourvu
8,9,5,Responsable Pédagogique,20000,28000,Confirmé,CDI,Ouvert
9,10,5,Formateur IT,18000,25000,Confirmé,CDD,Ouvert


In [8]:
pd.read_sql_query("SELECT candidat_id, nom_candidat, localisation, niveau_etudes, annees_experience, statut_candidat FROM profil_candidat", conn)

,candidat_id,nom_candidat,localisation,niveau_etudes,annees_experience,statut_candidat
0,1,Ahmed Bennani,Casablanca,Master,5,Actif
1,2,Fatima Zahra Alaoui,Fes,Licence,3,Actif
2,3,Mohammed Karim,Rabat,Licence,1,Actif
3,4,Laila Saada,Marrakech,Master,8,Actif
4,5,Youssef Tahir,Agadir,Bac,0,Actif
5,6,Nadia Hassan,Casablanca,Master,6,Actif
6,7,Ibrahim Moussaoui,Rabat,Licence,2,Actif
7,8,Salima Bouazza,Fes,Licence,4,Actif
8,9,Khalid Amrani,Marrakech,Licence,2,Inactif
9,10,Zainab Bennani,Casablanca,Master,7,Actif


## 4. Les 5 questions

### Question 1 : Nombre d'offres par entreprise (JOIN + GROUP BY)

In [9]:
# 1. Nombre d'offres par entreprise (JOIN + GROUP BY)
# LEFT JOIN : une entreprise sans offre apparaît quand même, avec 0
query = """
    SELECT
        e.nom_entreprise,
        COUNT(o.offre_id) AS nombre_offres
    FROM entreprise e
    LEFT JOIN offre o ON e.entreprise_id = o.entreprise_id
    GROUP BY e.entreprise_id, e.nom_entreprise
    ORDER BY nombre_offres DESC, e.nom_entreprise
"""
pd.read_sql_query(query, conn)

,nom_entreprise,nombre_offres
0,FinanceHub,3
1,TechVision Maroc,3
2,EduTech Solutions,2
3,MediCare Plus,2
4,RetailMax,2


**Réponse :** FinanceHub et TechVision Maroc ont chacune **3 offres** ; EduTech Solutions, MediCare Plus et RetailMax en ont **2** chacune, soit 12 offres au total. Le `LEFT JOIN` garantit qu'une entreprise sans offre apparaîtrait avec 0 : ici, aucune n'est dans ce cas.

### Question 2 : Salaire moyen par niveau d'expérience (GROUP BY + AVG)

In [10]:
# 2. Salaire moyen par niveau d'expérience (GROUP BY + AVG)
# salaire_moyen = moyenne du milieu de la fourchette (min + max) / 2
query = """
    SELECT
        niveau_experience,
        COUNT(*) AS nombre_offres,
        ROUND(AVG(salaire_min), 2) AS salaire_min_moyen,
        ROUND(AVG(salaire_max), 2) AS salaire_max_moyen,
        ROUND(AVG((salaire_min + salaire_max) / 2.0), 2) AS salaire_moyen
    FROM offre
    GROUP BY niveau_experience
    ORDER BY salaire_moyen DESC
"""
pd.read_sql_query(query, conn)

,niveau_experience,nombre_offres,salaire_min_moyen,salaire_max_moyen,salaire_moyen
0,Expert,2,29000.0,39000.0,34000.0
1,Confirmé,5,19400.0,26800.0,23100.0
2,Débutant,5,10200.0,13800.0,12000.0


**Réponse :** le salaire augmente avec l'expérience : **Expert 34 000**, **Confirmé 23 100**, **Débutant 12 000** (milieu de fourchette moyen). Un poste Expert paie environ 2,8 fois plus qu'un poste Débutant. Les niveaux Confirmé et Débutant comptent chacun 5 offres, Expert seulement 2.

### Question 3 : Candidats habitant dans une ville où il y a des offres ouvertes (JOIN + DISTINCT)

In [11]:
# 3. Candidats habitant dans une ville où il y a des offres ouvertes (JOIN + DISTINCT)
# La table offre n'a pas de ville : on prend la ville de l'entreprise qui publie l'offre
query = """
    SELECT DISTINCT
        c.nom_candidat,
        c.localisation,
        c.statut_candidat
    FROM profil_candidat c
    JOIN entreprise e ON c.localisation = e.localisation
    JOIN offre o ON o.entreprise_id = e.entreprise_id
    WHERE o.statut = 'Ouvert'
    ORDER BY c.localisation, c.nom_candidat
"""
result = pd.read_sql_query(query, conn)
print(f"{len(result)} candidats habitent dans une ville avec des offres ouvertes")
result

13 candidats habitent dans une ville avec des offres ouvertes


,nom_candidat,localisation,statut_candidat
0,Ahmed Bennani,Casablanca,Actif
1,Aisha Farraj,Casablanca,Actif
2,Nadia Hassan,Casablanca,Actif
3,Zainab Bennani,Casablanca,Actif
4,Fatima Zahra Alaoui,Fes,Actif
5,Salima Bouazza,Fes,Actif
6,Tariq El Fassi,Fes,Actif
7,Khalid Amrani,Marrakech,Inactif
8,Laila Saada,Marrakech,Actif
9,Sara Belkadi,Marrakech,Bloqué


**Réponse :** **13 candidats sur 15**. Seuls les deux candidats d'**Agadir** (Youssef Tahir et Rashid Mediane) sont exclus, car aucune entreprise n'y est installée.

Deux remarques :
- la table `offre` n'a pas de colonne ville : la ville d'une offre est celle de l'entreprise qui la publie ;
- le résultat inclut aussi les candidats « Inactif » (Khalid Amrani) et « Bloqué » (Sara Belkadi). Pour ne garder que les actifs, il suffit d'ajouter `AND c.statut_candidat = 'Actif'`.

### Question 4 : L'offre ayant le salaire maximum le plus élevé pour chaque type de contrat

In [12]:
# 4. L'offre ayant le salaire maximum le plus élevé pour chaque type de contrat
# Sous-requête corrélée : on garde l'offre dont salaire_max égale le maximum de son type de contrat
query = """
    SELECT
        o.type_contrat,
        o.titre_poste,
        e.nom_entreprise,
        o.salaire_max
    FROM offre o
    JOIN entreprise e ON o.entreprise_id = e.entreprise_id
    WHERE o.salaire_max = (
        SELECT MAX(o2.salaire_max)
        FROM offre o2
        WHERE o2.type_contrat = o.type_contrat
    )
    ORDER BY o.salaire_max DESC
"""
pd.read_sql_query(query, conn)

,type_contrat,titre_poste,nom_entreprise,salaire_max
0,CDI,Analyste Financier,FinanceHub,40000
1,CDD,Formateur IT,EduTech Solutions,25000
2,Stage,Stagiaire Finance,FinanceHub,8000


**Réponse :**
- **CDI** : Analyste Financier (FinanceHub), 40 000 ;
- **CDD** : Formateur IT (EduTech Solutions), 25 000 ;
- **Stage** : Stagiaire Finance (FinanceHub), 8 000.

En cas d'égalité de salaire maximum au sein d'un même type de contrat, cette requête renverrait toutes les offres à égalité.

### Question 5 : Secteurs d'activité des entreprises sans aucune offre d'emploi publiée (NOT IN)

In [13]:
# 5. Secteurs d'activité des entreprises sans aucune offre d'emploi publiée (NOT IN)
query = """
    SELECT DISTINCT secteur
    FROM entreprise
    WHERE entreprise_id NOT IN (
        SELECT entreprise_id FROM offre
    )
    ORDER BY secteur
"""
result = pd.read_sql_query(query, conn)
print(f"{len(result)} secteur(s) sans aucune offre publiée")
result

0 secteur(s) sans aucune offre publiée


,secteur


**Réponse :** le résultat est **vide**, ce qui est normal : les 5 entreprises ont toutes au moins une offre (voir la question 1). `offre.entreprise_id` est `NOT NULL`, donc le `NOT IN` n'est pas faussé par des valeurs nulles.

**Test de la requête :** on ajoute temporairement une entreprise sans offre pour vérifier qu'elle remonte bien, puis on la supprime.

In [14]:
# Test : une entreprise sans aucune offre
cursor.execute("""
    INSERT INTO entreprise (nom_entreprise, secteur, localisation, nombre_employes, site_web)
    VALUES ('AgriPlus', 'Agriculture', 'Agadir', 30, 'www.agriplus.ma')
""")
test = pd.read_sql_query(query, conn)

# Nettoyage : on retire l'entreprise de test
cursor.execute("DELETE FROM entreprise WHERE nom_entreprise = 'AgriPlus'")
conn.commit()
test

,secteur
0,Agriculture


Avec l'entreprise de test, la requête renvoie bien le secteur **Agriculture** : elle fonctionne.

## 5. Résumé des réponses

| Question | Réponse |
|---|---|
| 1. Offres par entreprise | FinanceHub 3, TechVision Maroc 3, EduTech Solutions 2, MediCare Plus 2, RetailMax 2 |
| 2. Salaire moyen par expérience | Expert 34 000, Confirmé 23 100, Débutant 12 000 |
| 3. Candidats dans une ville avec offres ouvertes | 13 sur 15 (Agadir exclu) |
| 4. Salaire max par type de contrat | CDI 40 000 (Analyste Financier), CDD 25 000 (Formateur IT), Stage 8 000 (Stagiaire Finance) |
| 5. Secteurs sans offre | Aucun (résultat vide) |